In [1]:
import pandas as pd

In [2]:
import sys

print(sys.executable)

C:\Users\daiyi\Documents\Github\MedAgent\.venv\Scripts\python.exe


In [12]:
path_to_data = "../data/raw/physionet.org/files/mimiciv/3.1/"
admissions = pd.read_csv(path_to_data + "hosp/admissions.csv.gz")
admissions.head()

,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,language,marital_status,race,edregtime,edouttime,hospital_expire_flag
0,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,English,WIDOWED,WHITE,2180-05-06 19:17:00,2180-05-06 23:30:00,0
1,10000032,22841357,2180-06-26 18:27:00,2180-06-27 18:49:00,NaN,EW EMER.,P784FA,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,WHITE,2180-06-26 15:54:00,2180-06-26 21:31:00,0
2,10000032,25742920,2180-08-05 23:44:00,2180-08-07 17:50:00,NaN,EW EMER.,P19UTS,EMERGENCY ROOM,HOSPICE,Medicaid,English,WIDOWED,WHITE,2180-08-05 20:58:00,2180-08-06 01:44:00,0
3,10000032,29079034,2180-07-23 12:35:00,2180-07-25 17:55:00,NaN,EW EMER.,P06OTX,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,WHITE,2180-07-23 05:54:00,2180-07-23 14:00:00,0
4,10000068,25022803,2160-03-03 23:16:00,2160-03-04 06:26:00,NaN,EU OBSERVATION,P39NWO,EMERGENCY ROOM,NaN,NaN,English,SINGLE,WHITE,2160-03-03 21:55:00,2160-03-04 06:26:00,0


In [13]:
admissions.info()
print(f"Number of rows: {admissions.size}")
print(pd.read_csv(path_to_data + "hosp/admissions.csv.gz").columns.tolist())

<class 'pandas.DataFrame'>
RangeIndex: 546028 entries, 0 to 546027
Data columns (total 16 columns):
 #   Column                Non-Null Count   Dtype
---  ------                --------------   -----
 0   subject_id            546028 non-null  int64
 1   hadm_id               546028 non-null  int64
 2   admittime             546028 non-null  str  
 3   dischtime             546028 non-null  str  
 4   deathtime             11790 non-null   str  
 5   admission_type        546028 non-null  str  
 6   admit_provider_id     546024 non-null  str  
 7   admission_location    546027 non-null  str  
 8   discharge_location    396210 non-null  str  
 9   insurance             536673 non-null  str  
 10  language              545253 non-null  str  
 11  marital_status        532409 non-null  str  
 12  race                  546028 non-null  str  
 13  edregtime             379240 non-null  str  
 14  edouttime             379240 non-null  str  
 15  hospital_expire_flag  546028 non-null  int64


**Goal**

* Predict LOS (length of stay in days)
* Readmission risk (Bern: Will they come back in 30 days? Probability?)

In [ ]:
import pandas as pd

admissions = pd.read_csv(
    path_to_data + "hosp/admissions.csv.gz"
)  # in / out time for LOS calculation
patients = pd.read_csv(
    path_to_data + "hosp/patients.csv.gz"
)  # in / out time for LOS calculation
icustays = pd.read_csv(path_to_data + "icu/icustays.csv.gz")  # Predict readmission risk


print(f"Number of rows admissions: {admissions.size}")
print(f"Number of rows patients: {patients.size}")
print(f"Number of rows icustays: {icustays.size}")


print(pd.read_csv("hosp/admissions.csv.gz").columns.tolist())
print(pd.read_csv("hosp/patients.csv.gz").columns.tolist())
print(pd.read_csv("icu/icustays.csv.gz").columns.tolist())

# All three keys join on subject id

In [ ]:
df = admissions.merge(
    patients, on="subject_id", how="inner"
)  # inner keeps information and rows for LOS prediction and readmission risk prediction
df = df.merge(icustays, on=["subject_id", "hadm_id"], how="inner")

In [ ]:
df.head()

In [ ]:
print(f"Number of rows in merged dataset: {df.shape[0]}")

In [ ]:
print(f"admissions: {admissions.shape}")
print(f"patients:   {patients.shape}")
print(f"icustays:   {icustays.shape}")
print(f"merged df:  {df.shape}")

**Explore distribution of LOS and readmission sample space**

In [ ]:
df.head(1)

In [ ]:
print(df.columns.tolist())

In [ ]:
df.columns

In [ ]:
for col in df.columns:
    print(f"{col}: {df[col].dtype} | sample: {df[col].dropna().iloc[0]}")

## Dataset overview

The merged dataframe combines three MIMIC-IV demo tables — `admissions`, `patients`, and `icustays` — 
joined on `subject_id` and `hadm_id`. It contains **100 demo patients** with **27 columns** spanning 
hospital admission records and ICU stay details.

**Key columns by category:**

- **IDs:** `subject_id` (patient), `hadm_id` (hospital admission), `stay_id` (ICU stay)
- **Admission timing:** `admittime`, `dischtime`, `edregtime`, `edouttime`
- **ICU timing:** `intime`, `outtime`, `los` (length of stay in days — our regression target)
- **Patient demographics:** `anchor_age`, `gender`, `race`, `language`, `marital_status`, `insurance`
- **Clinical:** `admission_type`, `first_careunit`, `last_careunit`, `hospital_expire_flag`
- **Mortality:** `deathtime`, `dod` (date of death), `hospital_expire_flag`

**Target variables:**
- `los` — continuous, length of ICU stay in days (for regression)
- Readmission — to be engineered from `admittime`/`dischtime` across multiple admissions

**Notes:**
- Dates are currently stored as strings — will need `pd.to_datetime()` conversion
- `anchor_year` is a shifted year for de-identification; actual calendar year is not meaningful
- `dod` and `deathtime` are present but sparse (only patients who died)

In [ ]:
date_cols = [
    "admittime",
    "dischtime",
    "edregtime",
    "edouttime",
    "intime",
    "outtime",
    "deathtime",
    "dod",
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")


df.head()
print(df.shape)

In [ ]:
df.head()

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df["los"], bins=40, color="steelblue", alpha=0.7)
ax.set_xlabel("Length of stay (days)")
ax.set_ylabel("Count")
ax.set_title("ICU length of stay distribution")
plt.tight_layout()
plt.show()

print(df["los"].describe())

LOS looks like an exponential distribution with 

* count RV: 140
* mean: 3.6
* std: 3.89